In [8]:
import re
from pathlib import Path
import pandas as pd

# ========== CONFIG ==========
HISTORY_ROOT = Path(r"week_folders")          # parent folder containing week folders
MASTER_HISTORY = Path("historical_matchups.csv")

# ========== Parsing helpers ==========
ROUND_FILE_RE = re.compile(r"(?i)^(?:r|round)\s*(\d+)\.txt$")

LINE_RE = re.compile(
    r"""^
    \s*(?P<table>\d*)\s+
    (?P<player>.+?)\s{2,}
    (?P<opponent>.+?)\s{2,}
    (?P<points>\d+\s*-\s*\d+|\d+)\s*
    $""",
    re.VERBOSE,
)

def find_round_files(folder: Path) -> list[tuple[int, Path]]:
    found = []
    for p in folder.iterdir():
        if p.is_file() and p.suffix.lower() == ".txt":
            m = ROUND_FILE_RE.match(p.name)
            if m:
                found.append((int(m.group(1)), p))
    if not found:
        raise FileNotFoundError(f"No round files found in {folder} (expected r1.txt, r2.txt, ...).")
    return sorted(found, key=lambda x: x[0])

def split_totals(points_raw: str) -> tuple[int, int | None]:
    """
    Handles '9-6' => (9, 6) and '6' => (6, None) for byes.
    """
    s = str(points_raw).replace(" ", "")
    if "-" in s:
        a, b = s.split("-", 1)
        return int(a), int(b)
    return int(s), None

def parse_round_file(path: Path, round_num: int) -> list[dict]:
    rows = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.rstrip()

        if not line.strip():
            continue
        if "Table" in line or set(line.strip()) <= {"-"}:
            continue
        if line.strip().lower().startswith("round"):
            continue

        m = LINE_RE.match(line)
        if not m:
            continue

        table = m.group("table").strip() or None
        player = m.group("player").strip()
        opponent = m.group("opponent").strip()
        points_raw = m.group("points").replace(" ", "")

        player_total, opp_total = split_totals(points_raw)

        rows.append({
            "Round": round_num,
            "Table": table,
            "Player": player,
            "Opponent": opponent,
            "PointsRaw": points_raw,
            "PlayerTotal": player_total,
            "OpponentTotal": opp_total,              # <-- NEW
            "IsBye": (opponent.strip() == "***Bye***"),
        })
    return rows

def build_matches_df(week_folder: Path) -> pd.DataFrame:
    recs = []
    for rnd, path in find_round_files(week_folder):
        recs.extend(parse_round_file(path, rnd))
    df = pd.DataFrame(recs)
    if df.empty:
        raise ValueError("Parsed 0 rows. Check formatting/regex.")
    return df.sort_values(["Round", "Table", "Player"]).reset_index(drop=True)

def interpret_results(df_matches: pd.DataFrame) -> pd.DataFrame:
    """
    FIXED for your data:
      - We DOUBLE rows so every player gets totals updated (player + opponent perspective)
      - BYE rows are special: PointsRaw shows "6" every time, NOT cumulative.
        So for byes we set NewTotal = PrevTotal + 3 (and Gain=3) regardless of the parsed "6".
    """
    required = {"Round", "Player", "Opponent", "PlayerTotal", "OpponentTotal", "IsBye", "PointsRaw"}
    missing = required - set(df_matches.columns)
    if missing:
        raise KeyError(f"interpret_results missing columns: {sorted(missing)}")

    prev_totals: dict[str, int] = {}
    out: list[dict] = []

    for rnd in sorted(df_matches["Round"].unique()):
        df_r = df_matches[df_matches["Round"] == rnd]

        for _, row in df_r.iterrows():
            table = row["Table"]
            p = str(row["Player"]).strip()
            o = str(row["Opponent"]).strip()
            is_bye = bool(row["IsBye"])

            # -----------------------
            # PLAYER SIDE (always)
            # -----------------------
            p_prev = int(prev_totals.get(p, 0))

            if is_bye:
                # ✅ CRITICAL FIX:
                # Bye PointsRaw is always "6" in your files, so DO NOT treat it as total.
                p_gain = 3
                p_new = p_prev + 3
                p_outcome = "Bye"
            else:
                # for non-byes, PlayerTotal from the file *is* cumulative
                p_new = int(row["PlayerTotal"])
                p_gain = p_new - p_prev
                if p_gain == 3:
                    p_outcome = "Win"
                elif p_gain == 1:
                    p_outcome = "Tie"
                elif p_gain == 0:
                    p_outcome = "Loss"
                else:
                    p_outcome = f"Check({p_gain})"

            prev_totals[p] = p_new

            out.append({
                "Round": int(rnd),
                "Table": table,
                "Player": p,
                "Opponent": o,
                "PrevTotal": p_prev,
                "NewTotal": p_new,
                "Gain": p_gain,
                "Outcome": p_outcome,
                "IsBye": is_bye,
                "Source": "PlayerRow",
                "PointsRaw": row["PointsRaw"],
            })

            # -----------------------
            # OPPONENT SIDE (only if not bye)
            # -----------------------
            if not is_bye:
                # OpponentTotal is the opponent's cumulative total from the same X-Y entry
                o_new = row["OpponentTotal"]
                o_new = int(o_new) if pd.notna(o_new) else None

                if o_new is None:
                    # can't invert without opponent total
                    out.append({
                        "Round": int(rnd),
                        "Table": table,
                        "Player": o,
                        "Opponent": p,
                        "PrevTotal": int(prev_totals.get(o, 0)),
                        "NewTotal": None,
                        "Gain": None,
                        "Outcome": "NeedsReview(MissingOpponentTotal)",
                        "IsBye": False,
                        "Source": "InvertedOpponentRow",
                        "PointsRaw": row["PointsRaw"],
                    })
                else:
                    o_prev = int(prev_totals.get(o, 0))
                    o_gain = o_new - o_prev

                    if o_gain == 3:
                        o_outcome = "Win"
                    elif o_gain == 1:
                        o_outcome = "Tie"
                    elif o_gain == 0:
                        o_outcome = "Loss"
                    else:
                        o_outcome = f"Check({o_gain})"

                    prev_totals[o] = o_new

                    out.append({
                        "Round": int(rnd),
                        "Table": table,
                        "Player": o,
                        "Opponent": p,
                        "PrevTotal": o_prev,
                        "NewTotal": o_new,
                        "Gain": o_gain,
                        "Outcome": o_outcome,
                        "IsBye": False,
                        "Source": "InvertedOpponentRow",
                        "PointsRaw": row["PointsRaw"],
                    })

    df_out = pd.DataFrame(out).sort_values(["Round", "Table", "Player", "Source"]).reset_index(drop=True)

    # Optional sanity print: bye gains should all be 3
    bad_byes = df_out[(df_out["IsBye"] == True) & (df_out["Gain"] != 3)]
    if not bad_byes.empty:
        print("[WARN] Some bye rows did not get Gain=3 (should never happen now).")

    return df_out

# ========== Matchups from interpreted results (expects both directions now) ==========
def build_week_matchups(df_results: pd.DataFrame, week_id: str) -> pd.DataFrame:
    """
    Now that interpret_results produces BOTH directions, we can safely pair them and compute ScoreA.
    """
    df = df_results[~df_results["IsBye"]].copy()

    def key(rnd, a, b):
        lo, hi = sorted([a.strip(), b.strip()])
        return (int(rnd), lo, hi)

    df["MatchKey"] = [key(r, a, b) for r, a, b in zip(df["Round"], df["Player"], df["Opponent"])]

    rows = []
    map_score = {"Win": 1.0, "Tie": 0.5, "Loss": 0.0}

    for (rnd, lo, hi), g in df.groupby("MatchKey"):
        if len(g) != 2:
            rows.append({
                "Week": week_id,
                "Round": int(rnd),
                "PlayerA": lo,
                "PlayerB": hi,
                "ScoreA": None,
                "OutcomeA": None,
                "OutcomeB": None,
                "Status": f"Unpaired({len(g)})",
                "MatchKey": f"{rnd}|{lo}|{hi}",
            })
            continue

        row_lo = g[(g["Player"] == lo) & (g["Opponent"] == hi)]
        row_hi = g[(g["Player"] == hi) & (g["Opponent"] == lo)]

        if len(row_lo) != 1 or len(row_hi) != 1:
            rows.append({
                "Week": week_id,
                "Round": int(rnd),
                "PlayerA": lo,
                "PlayerB": hi,
                "ScoreA": None,
                "OutcomeA": None,
                "OutcomeB": None,
                "Status": "BadPairingRows",
                "MatchKey": f"{rnd}|{lo}|{hi}",
            })
            continue

        o_lo = str(row_lo["Outcome"].iloc[0]).strip()
        o_hi = str(row_hi["Outcome"].iloc[0]).strip()

        score_a = map_score.get(o_lo, None)

        expected_other = {"Win": "Loss", "Loss": "Win", "Tie": "Tie"}.get(o_lo)
        ok = (score_a is not None) and (expected_other is not None) and (o_hi == expected_other)

        rows.append({
            "Week": week_id,
            "Round": int(rnd),
            "PlayerA": lo,
            "PlayerB": hi,
            "ScoreA": score_a,
            "OutcomeA": o_lo,
            "OutcomeB": o_hi,
            "Status": "OK" if ok else "CheckOutcomeMismatch",
            "MatchKey": f"{rnd}|{lo}|{hi}",
        })

    return pd.DataFrame(rows).sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

# ========== Ingest week + update master ==========
def ingest_week_and_update_master(week_folder: Path, week_id: str) -> tuple[Path, Path]:
    week_folder.mkdir(parents=True, exist_ok=True)

    df_matches = build_matches_df(week_folder)
    df_results = interpret_results(df_matches)
    df_matchups = build_week_matchups(df_results, week_id=week_id)

    week_results_path = week_folder / f"{week_id}_results_interpreted.csv"
    week_matchups_path = week_folder / f"{week_id}_matchups.csv"

    df_results.to_csv(week_results_path, index=False)
    df_matchups.to_csv(week_matchups_path, index=False)

    # upsert into master (replace week)
    df_week = df_matchups.copy()
    df_week["Week"] = df_week["Week"].astype(str)

    if MASTER_HISTORY.exists():
        df_master = pd.read_csv(MASTER_HISTORY)
        df_master["Week"] = df_master["Week"].astype(str)
        df_master = df_master[df_master["Week"] != str(week_id)].copy()
        df_master = pd.concat([df_master, df_week], ignore_index=True)
    else:
        df_master = df_week

    df_master["Round"] = df_master["Round"].astype(int)
    df_master = df_master.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)
    df_master.to_csv(MASTER_HISTORY, index=False)

    print(f"[WEEK] wrote: {week_results_path.name}, {week_matchups_path.name}")
    print(f"[MASTER] updated: {MASTER_HISTORY}")
    print("[WEEK] matchup status:\n", df_matchups["Status"].value_counts(dropna=False).to_string())

    # Helpful sanity: show any remaining Check(...) rows
    bad = df_results[df_results["Outcome"].astype(str).str.startswith("Check(")]
    if not bad.empty:
        print(f"[WARN] {len(bad)} interpreted rows have Check(...). Sample:")
        print(bad.head(10).to_string(index=False))

    return week_results_path, week_matchups_path

# ========== RUN THIS ONCE PER WEEK ==========
if __name__ == "__main__":
    week_id = "2_20_26"
    week_folder = HISTORY_ROOT / week_id
    ingest_week_and_update_master(week_folder, week_id)

[WEEK] wrote: 2_20_26_results_interpreted.csv, 2_20_26_matchups.csv
[MASTER] updated: historical_matchups.csv
[WEEK] matchup status:
 Status
OK    42


In [17]:
from pathlib import Path
import pandas as pd

import pandas as pd
from pathlib import Path

# =========================
# 1) Load player_list.csv -> alias map + opt-in set
# =========================
def load_player_list(player_list_csv: str | Path) -> tuple[dict[str, str], set[str]]:
    """
    player_list.csv format:
      canonical_name, alias1, alias2, alias3, ...
    Returns:
      alias_to_canonical: dict mapping any alias/canonical -> canonical
      opt_in_canonicals: set of canonical names (column 1 values)
    """
    df = pd.read_csv(player_list_csv, header=None, dtype=str).fillna("")
    alias_to_canonical: dict[str, str] = {}
    opt_in: set[str] = set()

    for _, row in df.iterrows():
        canonical = str(row.iloc[0]).strip()
        if not canonical:
            continue
        opt_in.add(canonical)

        # Map canonical to itself
        alias_to_canonical[canonical] = canonical

        # Map aliases to canonical
        for cell in row.iloc[1:]:
            alias = str(cell).strip()
            if alias:
                alias_to_canonical[alias] = canonical

    return alias_to_canonical, opt_in


def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    n = str(name).strip()
    return alias_to_canonical.get(n, n)


# =========================
# 2) Elo helpers
# =========================
def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float) -> tuple[float, float]:
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))


# =========================
# 3) Elo from historical record with:
#    - alias collapsing
#    - opt-in tracking only
#    - non-opt-in opponents always treated as 1500 and NOT updated
# =========================
def run_opt_in_elo_from_master(
    master_history_csv: str | Path,
    player_list_csv: str | Path,
    initial_elo: float = 1500.0,
    k: float = 32.0,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      leaderboard (final standings for opt-in players),
      match_history (only matches where at least one opt-in played),
      ladder_by_week (opt-in players only, snapshot each week; carries forward if inactive)
    """
    alias_to_canonical, opt_in = load_player_list(player_list_csv)

    df = pd.read_csv(master_history_csv)
    required = {"Week", "Round", "PlayerA", "PlayerB", "ScoreA", "Status"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"master history missing columns: {sorted(missing)}")

    # Only use OK matches
    df = df[df["Status"] == "OK"].copy()
    if df.empty:
        raise ValueError("No OK match rows found in master history.")

    # Canonicalize names
    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)

    # Deterministic order
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    # Ratings only for opt-in canonicals
    ratings: dict[str, float] = {p: float(initial_elo) for p in opt_in}
    stats: dict[str, dict[str, int]] = {p: {"Games": 0, "Wins": 0, "Losses": 0, "Ties": 0} for p in opt_in}

    # Weeks in data
    weeks = list(dict.fromkeys(df["Week"].tolist()))

    match_history_rows = []
    ladder_rows = []
    match_index = 0

    def get_rating_for_player(p: str) -> float:
        # opt-in: current rating; non-opt-in: always 1500
        return ratings[p] if p in opt_in else float(initial_elo)

    for wk in weeks:
        df_w = df[df["Week"] == wk]

        # Process matches in this week
        for _, m in df_w.iterrows():
            a = m["PlayerA"]
            b = m["PlayerB"]
            s_a = float(m["ScoreA"])

            a_in = (a in opt_in)
            b_in = (b in opt_in)

            # Skip match entirely if neither player is opt-in
            if not (a_in or b_in):
                continue

            pre_a = get_rating_for_player(a)
            pre_b = get_rating_for_player(b)

            post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

            # Update ONLY opt-in ratings
            if a_in:
                ratings[a] = post_a
            if b_in:
                ratings[b] = post_b

            # Update ONLY opt-in stats
            if a_in:
                stats[a]["Games"] += 1
                if s_a == 1.0:
                    stats[a]["Wins"] += 1
                elif s_a == 0.0:
                    stats[a]["Losses"] += 1
                else:
                    stats[a]["Ties"] += 1

            if b_in:
                stats[b]["Games"] += 1
                s_b = 1.0 - s_a
                if s_b == 1.0:
                    stats[b]["Wins"] += 1
                elif s_b == 0.0:
                    stats[b]["Losses"] += 1
                else:
                    stats[b]["Ties"] += 1

            match_index += 1
            match_history_rows.append({
                "MatchIndex": match_index,
                "Week": wk,
                "Round": int(m["Round"]),
                "PlayerA": a,
                "PlayerB": b,
                "ScoreA": s_a,
                "PreA": pre_a,
                "PreB": pre_b,
                "PostA": post_a if a_in else None,   # None because we don't track non-opt-in updates
                "PostB": post_b if b_in else None,
                "A_OptIn": a_in,
                "B_OptIn": b_in,
                "K": k,
                "NonOptInOppRatingUsed": initial_elo,  # explicit
            })

        # Snapshot: EVERY opt-in player each week (even if they didn't play)
        for p in sorted(opt_in):
            ladder_rows.append({
                "Week": wk,
                "Player": p,
                "Elo": ratings.get(p, float(initial_elo)),
                "Games": stats[p]["Games"],
                "Wins": stats[p]["Wins"],
                "Losses": stats[p]["Losses"],
                "Ties": stats[p]["Ties"],
            })

    match_history = pd.DataFrame(match_history_rows)
    ladder_by_week = pd.DataFrame(ladder_rows).sort_values(["Player", "Week"]).reset_index(drop=True)

    ladder_by_week["DeltaElo"] = ladder_by_week.groupby("Player")["Elo"].diff().fillna(0.0)
    ladder_by_week["Elo"] = ladder_by_week["Elo"].round(2)
    ladder_by_week["DeltaElo"] = ladder_by_week["DeltaElo"].round(2)

    # Final leaderboard = last week snapshot
    last_week = weeks[-1]
    leaderboard = (
        ladder_by_week[ladder_by_week["Week"] == last_week]
        .sort_values(["Elo", "Wins"], ascending=[False, False])
        .reset_index(drop=True)
    )

    return leaderboard, match_history, ladder_by_week


def print_leaders(leaderboard: pd.DataFrame, top_n: int = 15):
    cols = ["Player", "Elo", "Games", "Wins", "Losses", "Ties"]
    print(leaderboard[cols].head(top_n).to_string(index=False))


# ==========
# RUN THIS ANYTIME (RECOMPUTES FROM SCRATCH)
# ==========
MASTER_HISTORY = "historical_matchups.csv"
PLAYER_LIST = "player_list.csv"

leaderboard, match_history, ladder_by_week = run_opt_in_elo_from_master(
    master_history_csv=MASTER_HISTORY,
    player_list_csv=PLAYER_LIST,
    initial_elo=1500,
    k=32,
)

print_leaders(leaderboard, top_n=50)

leaderboard.to_csv("elo_leaderboard.csv", index=False)
match_history.to_csv("elo_match_history.csv", index=False)
ladder_by_week.to_csv("elo_ladder_by_week.csv", index=False)

           Player     Elo  Games  Wins  Losses  Ties
Anthony Wilkinson 1596.51     10     8       1     1
   David Van Hise 1587.37     10     8       2     0
    Stan Kornacki 1552.93     10     7       2     1
   Nathan Cormier 1547.66      5     4       1     0
       Amy Altman 1543.33     10     6       3     1
       Milo Brown 1531.29      5     3       1     1
      Nathan Choi 1529.93      5     3       1     1
  John Larochelle 1527.88      5     3       1     1
Isharah Considine 1516.55      5     3       2     0
     Amber Cronin 1516.03      5     3       2     0
       Mike Adams 1507.91     10     5       4     1
  Becca Blackwood 1507.65      9     4       4     1
    Bailey Sarkis 1500.00      0     0       0     0
   Bradley Skubic 1500.00      0     0       0     0
    Elias Medrano 1500.00      0     0       0     0
      Logan Hesch 1500.00      0     0       0     0
          Ran Guo 1500.00      0     0       0     0
       Dani Masom 1490.87      9     4       5

In [18]:
import pandas as pd
from pathlib import Path
import json

# =========================
# CONFIG (EDIT THESE)
# =========================
MASTER_HISTORY_CSV = Path("historical_matchups.csv")     # your master matchups
PLAYER_LIST_CSV    = Path("player_list.csv")            # canonical,alias1,alias2,...
OUT_LEADERBOARD_JSON = Path("site/data/leaderboard.json")

INITIAL_ELO = 1500.0
K_FACTOR = 32.0


# =========================
# Player list -> alias map
# =========================
def load_player_list(player_list_csv: Path) -> tuple[dict[str, str], set[str]]:
    df = pd.read_csv(player_list_csv, header=None, dtype=str).fillna("")
    alias_to_canonical: dict[str, str] = {}
    opt_in: set[str] = set()

    for _, row in df.iterrows():
        canonical = str(row.iloc[0]).strip()
        if not canonical:
            continue
        opt_in.add(canonical)
        alias_to_canonical[canonical] = canonical
        for cell in row.iloc[1:]:
            alias = str(cell).strip()
            if alias:
                alias_to_canonical[alias] = canonical

    return alias_to_canonical, opt_in

def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    n = str(name).strip()
    return alias_to_canonical.get(n, n)


# =========================
# Elo helpers
# =========================
def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float) -> tuple[float, float]:
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))


# =========================
# Main: compute opt-in Elo + export leaderboard.json
# =========================
def export_leaderboard_json(
    master_history_csv: Path,
    player_list_csv: Path,
    out_json: Path,
    initial_elo: float = 1500.0,
    k: float = 32.0,
):
    alias_to_canonical, opt_in = load_player_list(player_list_csv)

    df = pd.read_csv(master_history_csv)

    required = {"Week", "Round", "PlayerA", "PlayerB", "ScoreA", "Status"}
    missing = required - set(df.columns)
    if missing:
        raise KeyError(f"master_history missing columns: {sorted(missing)}")

    # Only OK matches (byes already excluded upstream)
    df = df[df["Status"] == "OK"].copy()
    if df.empty:
        raise ValueError("No OK matches found in master history. Check Status values and matchup generation.")

    # Canonicalize names
    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)

    # Deterministic order
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    # Ratings/stats only for opt-in players
    ratings = {p: float(initial_elo) for p in opt_in}
    stats = {p: {"games": 0, "wins": 0, "losses": 0, "ties": 0} for p in opt_in}

    def rating_for(p: str) -> float:
        # Opt-in: current rating; non-opt-in: fixed neutral
        return ratings[p] if p in opt_in else float(initial_elo)

    # Iterate matches
    for _, m in df.iterrows():
        a = m["PlayerA"]
        b = m["PlayerB"]
        s_a = float(m["ScoreA"])

        a_in = a in opt_in
        b_in = b in opt_in

        # Ignore matches where neither is opt-in
        if not (a_in or b_in):
            continue

        pre_a = rating_for(a)
        pre_b = rating_for(b)

        post_a, post_b = elo_update(pre_a, pre_b, s_a, k)

        # Update ONLY opt-in ratings
        if a_in:
            ratings[a] = post_a
        if b_in:
            ratings[b] = post_b

        # Update ONLY opt-in stats
        if a_in:
            stats[a]["games"] += 1
            if s_a == 1.0:
                stats[a]["wins"] += 1
            elif s_a == 0.0:
                stats[a]["losses"] += 1
            else:
                stats[a]["ties"] += 1

        if b_in:
            stats[b]["games"] += 1
            s_b = 1.0 - s_a
            if s_b == 1.0:
                stats[b]["wins"] += 1
            elif s_b == 0.0:
                stats[b]["losses"] += 1
            else:
                stats[b]["ties"] += 1

    # Build leaderboard rows in the exact format your HTML expects
    rows = []
    for p in sorted(opt_in):
        rows.append({
            "player": p,
            "elo": round(float(ratings.get(p, initial_elo)), 2),
            "games": int(stats[p]["games"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "ties": int(stats[p]["ties"]),
        })

    # Sort by Elo desc, wins desc
    rows.sort(key=lambda r: (r["elo"], r["wins"]), reverse=True)

    out_json.parent.mkdir(parents=True, exist_ok=True)
    with open(out_json, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

    print(f"✅ Wrote {out_json} ({len(rows)} players)")
    print("Top 10:")
    for r in rows[:10]:
        print(f"  {r['player']:<24} Elo={r['elo']:<7} W-L-T={r['wins']}-{r['losses']}-{r['ties']}  Games={r['games']}")


# =========================
# RUN
# =========================
if __name__ == "__main__":
    export_leaderboard_json(
        master_history_csv=MASTER_HISTORY_CSV,
        player_list_csv=PLAYER_LIST_CSV,
        out_json=OUT_LEADERBOARD_JSON,
        initial_elo=INITIAL_ELO,
        k=K_FACTOR,
    )

✅ Wrote site\data\leaderboard.json (25 players)
Top 10:
  Anthony Wilkinson        Elo=1596.51 W-L-T=8-1-1  Games=10
  David Van Hise           Elo=1587.37 W-L-T=8-2-0  Games=10
  Stan Kornacki            Elo=1552.93 W-L-T=7-2-1  Games=10
  Nathan Cormier           Elo=1547.66 W-L-T=4-1-0  Games=5
  Amy Altman               Elo=1543.33 W-L-T=6-3-1  Games=10
  Milo Brown               Elo=1531.29 W-L-T=3-1-1  Games=5
  Nathan Choi              Elo=1529.93 W-L-T=3-1-1  Games=5
  John Larochelle          Elo=1527.88 W-L-T=3-1-1  Games=5
  Isharah Considine        Elo=1516.55 W-L-T=3-2-0  Games=5
  Amber Cronin             Elo=1516.03 W-L-T=3-2-0  Games=5


In [19]:
import pandas as pd
from pathlib import Path
import json

# =========================
# CONFIG (EDIT)
# =========================
MASTER_HISTORY_CSV = Path("historical_matchups.csv")
PLAYER_LIST_CSV = Path("player_list.csv")

SITE_DATA_DIR = Path("site/data")
LEADERBOARD_JSON = SITE_DATA_DIR / "leaderboard.json"
PLAYERS_DIR = SITE_DATA_DIR / "players"
PLAYERS_INDEX_JSON = SITE_DATA_DIR / "players_index.json"

INITIAL_ELO = 1500.0
K_FACTOR = 32.0

# =========================
# Helpers
# =========================
def slugify(name: str) -> str:
    return (
        str(name).lower().strip()
        .replace("'", "").replace('"', "")
        .translate({ord(c): "-" for c in " _./\\|,:;()[]{}!"})
        .replace("--", "-").strip("-")
    )

def load_player_list(player_list_csv: Path) -> tuple[dict[str, str], set[str]]:
    df = pd.read_csv(player_list_csv, header=None, dtype=str).fillna("")
    alias_to_canonical = {}
    opt_in = set()
    for _, row in df.iterrows():
        canonical = str(row.iloc[0]).strip()
        if not canonical:
            continue
        opt_in.add(canonical)
        alias_to_canonical[canonical] = canonical
        for cell in row.iloc[1:]:
            alias = str(cell).strip()
            if alias:
                alias_to_canonical[alias] = canonical
    return alias_to_canonical, opt_in

def canon_name(name: str, alias_to_canonical: dict[str, str]) -> str:
    n = str(name).strip()
    return alias_to_canonical.get(n, n)

def expected_score(r_a: float, r_b: float) -> float:
    return 1.0 / (1.0 + 10 ** ((r_b - r_a) / 400.0))

def elo_update(r_a: float, r_b: float, s_a: float, k: float):
    e_a = expected_score(r_a, r_b)
    e_b = 1.0 - e_a
    s_b = 1.0 - s_a
    return (r_a + k * (s_a - e_a), r_b + k * (s_b - e_b))

# =========================
# Build site data
# =========================
def export_site_data():
    alias_to_canonical, opt_in = load_player_list(PLAYER_LIST_CSV)

    df = pd.read_csv(MASTER_HISTORY_CSV)
    df = df[df["Status"] == "OK"].copy()
    if df.empty:
        raise ValueError("No OK matches found in historical_matchups.csv")

    df["Week"] = df["Week"].astype(str)
    df["Round"] = df["Round"].astype(int)
    df["PlayerA"] = df["PlayerA"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["PlayerB"] = df["PlayerB"].astype(str).map(lambda x: canon_name(x, alias_to_canonical))
    df["ScoreA"] = df["ScoreA"].astype(float)
    df = df.sort_values(["Week", "Round", "PlayerA", "PlayerB"]).reset_index(drop=True)

    ratings = {p: float(INITIAL_ELO) for p in opt_in}
    stats = {p: {"games": 0, "wins": 0, "losses": 0, "ties": 0} for p in opt_in}

    player_matches = {p: [] for p in opt_in}  # list of match dicts per player

    def r(p):  # rating used for expectation
        return ratings[p] if p in opt_in else float(INITIAL_ELO)

    for _, m in df.iterrows():
        a = m["PlayerA"]; b = m["PlayerB"]; s_a = float(m["ScoreA"])
        a_in = a in opt_in
        b_in = b in opt_in

        if not (a_in or b_in):
            continue

        pre_a = r(a); pre_b = r(b)
        post_a, post_b = elo_update(pre_a, pre_b, s_a, K_FACTOR)

        # Update only opt-in
        if a_in: ratings[a] = post_a
        if b_in: ratings[b] = post_b

        # Stats + per-player history rows
        def add_row(player, opponent, score, pre, post):
            if player not in opt_in:
                return
            # result from player's perspective
            res = "W" if score == 1.0 else "L" if score == 0.0 else "T"
            player_matches[player].append({
                "week": m["Week"],
                "round": int(m["Round"]),
                "opponent": opponent,
                "result": res,
                "score": score,
                "elo_before": round(float(pre), 4),
                "elo_after": round(float(post), 4),
            })

        # Update stats for opt-in players only
        if a_in:
            stats[a]["games"] += 1
            if s_a == 1.0: stats[a]["wins"] += 1
            elif s_a == 0.0: stats[a]["losses"] += 1
            else: stats[a]["ties"] += 1
            add_row(a, b, s_a, pre_a, post_a)

        if b_in:
            s_b = 1.0 - s_a
            stats[b]["games"] += 1
            if s_b == 1.0: stats[b]["wins"] += 1
            elif s_b == 0.0: stats[b]["losses"] += 1
            else: stats[b]["ties"] += 1
            add_row(b, a, s_b, pre_b, post_b)

    # Ensure output dirs
    SITE_DATA_DIR.mkdir(parents=True, exist_ok=True)
    PLAYERS_DIR.mkdir(parents=True, exist_ok=True)

    # Leaderboard JSON
    leaderboard_rows = []
    for p in sorted(opt_in):
        leaderboard_rows.append({
            "player": p,
            "elo": round(float(ratings.get(p, INITIAL_ELO)), 2),
            "games": int(stats[p]["games"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "ties": int(stats[p]["ties"]),
        })
    leaderboard_rows.sort(key=lambda r: (r["elo"], r["wins"]), reverse=True)
    LEADERBOARD_JSON.write_text(json.dumps(leaderboard_rows, ensure_ascii=False, indent=2), encoding="utf-8")

    # Per-player JSON + index
    players_index = []
    for p in sorted(opt_in):
        slug = slugify(p)
        pdata = {
            "player": p,
            "slug": slug,
            "elo": round(float(ratings.get(p, INITIAL_ELO)), 2),
            "games": int(stats[p]["games"]),
            "wins": int(stats[p]["wins"]),
            "losses": int(stats[p]["losses"]),
            "ties": int(stats[p]["ties"]),
            "matches": player_matches.get(p, []),
        }
        (PLAYERS_DIR / f"{slug}.json").write_text(json.dumps(pdata, ensure_ascii=False, indent=2), encoding="utf-8")
        players_index.append({"player": p, "slug": slug})

    PLAYERS_INDEX_JSON.write_text(json.dumps(players_index, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"✅ wrote {LEADERBOARD_JSON}")
    print(f"✅ wrote {PLAYERS_INDEX_JSON}")
    print(f"✅ wrote {len(players_index)} player JSON files into {PLAYERS_DIR}/")

if __name__ == "__main__":
    export_site_data()

✅ wrote site\data\leaderboard.json
✅ wrote site\data\players_index.json
✅ wrote 26 player JSON files into site\data\players/
